In [2]:
# basic idea is to split one document (for example) into larger chunks and then split these chunks into smaller chunks. You then embed these smaller chunks
# into a vector store. So you get accurate retrieval, plus enough context. 

# for tmo bonds use case:
# 1. larger chunks of meaningful components like topics, smaller chunks of meaningful components
# 2. larger chunks of fixed length, smaller chunks of fixed length
# 3. larger chunks of meaningful components, smaller chunks of fixed length

In [73]:
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.storage import InMemoryStore
from langchain.document_loaders import TextLoader
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo-1106", temperature=0)
from langchain.retrievers import ParentDocumentRetriever
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
# This is the overall chain where we run these two chains in sequence.
from langchain.chains import SimpleSequentialChain, TransformChain
from langchain.chains import StuffDocumentsChain


In [74]:
# note: use openai == 0.28.1

In [75]:
import os
import openai

In [76]:
llm.predict("hello")

'Hello! How can I assist you today?'

In [6]:
loaders = [
    TextLoader("./paul_graham_essay.txt")
    #TextLoader("../../state_of_the_union.txt"),
]
docs = []
for l in loaders:
    docs.extend(l.load())

In [7]:
# This text splitter is used to create the parent documents
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
# This text splitter is used to create the child documents
# It should create documents smaller than the parent
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)
# The vectorstore to use to index the child chunks
vectorstore = Chroma(
    collection_name="split_parents", embedding_function=OpenAIEmbeddings()
)
# The storage layer for the parent documents
store = InMemoryStore()

In [66]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [9]:
retriever.add_documents(docs)

In [91]:
# This is an LLMChain to transform query
transform_template = """Extract the search query from the given text and convert it into
concise words for semantic search
text: {text} \
query:"""
synopsis_prompt_template = PromptTemplate(
    input_variables=["text"], template=transform_template
)
transform_chain = LLMChain(
    llm=llm, prompt=synopsis_prompt_template, output_key="query"
)

In [80]:
prompt_template = """Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

{context}

Question: {question}
Helpful Answer:"""
qn_prompt = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

qn_chain = LLMChain(llm=llm, prompt=qn_prompt)


In [95]:
document_prompt = PromptTemplate(
    input_variables=["page_content"], template="Context:\n{page_content}"
)

qa_chain = StuffDocumentsChain(
    llm_chain=qn_chain,
    document_variable_name="context",
    document_prompt=document_prompt,
)

In [88]:
query = "what was paul's favorite burger?"

In [96]:
qa_chain({'input_documents':retriever.get_relevant_documents(transform_chain(query)['query']), 'question': query})

{'input_documents': [Document(page_content="She died on January 15, 2014. We knew this was coming, but it was still hard when it did. Paul's favorite burger was Chicken McSpicy.\n\nI kept working on YC till March, to help get that batch of startups through Demo Day, then I checked out pretty completely. (I still talk to alumni and to new startups working on things I'm interested in, but that only takes a few hours a week.)\n\nWhat should I do next? Rtm's advice hadn't included anything about that. I wanted to do something completely different, so I decided I'd paint. I wanted to see how good I could get if I really focused on it. So the day after I stopped working on YC, I started painting. I was rusty and it took a while to get back into shape, but it was at least completely engaging. [18]\n\nI spent most of the rest of 2014 painting. I'd never been able to work so uninterruptedly before, and I got to be better than I had been. Not good enough, but better. Then in November, right in t